# Level 1: create the Genie agent

Create the agent in the Genie UI first so you learn the product. Then this notebook creates the same blank and curated agents through the API using catalog-parameterized templates.


## Bind config

Loads your catalog, schema, and warehouse. Templates keep `${catalog}.${schema}` until this cell substitutes them.


In [ ]:
import sys
from pathlib import Path

dbutils.widgets.text("catalog", "YOUR_CATALOG", "Catalog")
dbutils.widgets.text("schema", "YOUR_SCHEMA", "Schema")
dbutils.widgets.text("warehouse_id", "YOUR_WAREHOUSE_ID", "SQL warehouse id")
dbutils.widgets.text("repo_root", "", "Workspace folder that contains shared/")

here = Path.cwd().resolve()
root = None
repo_root_w = dbutils.widgets.get("repo_root").strip()
if repo_root_w and (Path(repo_root_w) / "shared" / "config.py").exists():
    root = Path(repo_root_w)
if root is None:
    for candidate in [here, *here.parents]:
        if (candidate / "shared" / "config.py").exists():
            root = candidate
            break
if root is None:
    raise FileNotFoundError("shared/config.py not found. Set repo_root to the workspace folder that contains shared/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from shared.config import load_config
from shared.lib.genie_client import GenieClient

cfg = load_config()
print("catalog", cfg.catalog)
print("schema", cfg.schema)
print("fqn", cfg.fqn)


## Load blank and curated templates

Templates live in `level-1-foundations/templates/`. Placeholders are replaced with the widget values.


In [ ]:
import json
from pathlib import Path

tpl_dir = root / "level-1-foundations" / "templates"

def load_tpl(name: str) -> dict:
    raw = (tpl_dir / name).read_text()
    raw = raw.replace("${catalog}", cfg.catalog).replace("${schema}", cfg.schema)
    return json.loads(raw)

blank = load_tpl("agent_blank.json")
curated = load_tpl("agent_curated.json")
print("blank tables", len(blank["table_identifiers"]))
print("curated examples", len(curated["example_question_sqls"]))
print("curated benchmarks", len(curated["benchmarks"]))


## Create both agents

The API path is still `/api/2.0/genie/spaces`. Reuses an existing agent with the same title in this workspace when found.


In [ ]:
client = GenieClient(warehouse_id=cfg.warehouse_id)
existing = {s.get("title"): s for s in client.list_spaces()}

def ensure(title: str, description: str, tables: list[str]) -> dict:
    found = existing.get(title)
    if found:
        print("reuse", title, found.get("space_id") or found.get("id"))
        return found
    created = client.create_space(title, description, tables)
    print("created", title, created.get("space_id") or created.get("id"))
    return created

blank_space = ensure(blank["title"], blank["description"], blank["table_identifiers"])
curated_space = ensure(curated["title"], curated["description"], curated["table_identifiers"])
blank_id = blank_space.get("space_id") or blank_space.get("id")
curated_id = curated_space.get("space_id") or curated_space.get("id")
assert blank_id and curated_id
print("blank url", client.ui_space_url(blank_id))
print("curated url", client.ui_space_url(curated_id))


## GET-merge-PATCH the curated instructions

Updates `text_instructions`, sorted `example_question_sqls`, and benchmark questions. Never PATCH a payload you did not GET first.


In [ ]:
def apply_curated(space: dict) -> dict:
    instructions = space.setdefault("instructions", {})
    instructions["text_instructions"] = [
        {"id": "l1-text-01", "content": [curated["text_instructions"]]}
    ]
    instructions["example_question_sqls"] = curated["example_question_sqls"]
    questions = []
    for item in curated["benchmarks"]:
        questions.append(
            {
                "id": item["id"],
                "question": item["question"],
                "answer": [{"format": "SQL", "content": item["sql"]}],
            }
        )
    space["benchmarks"] = {"questions": questions}
    return space

patched = client.merge_serialized_space(curated_id, apply_curated)
blob = patched.get("serialized_space") or client.get_space(curated_id).get("serialized_space")
if isinstance(blob, str):
    blob = json.loads(blob)
examples = ((blob.get("instructions") or {}).get("example_question_sqls")) or []
bqs = ((blob.get("benchmarks") or {}).get("questions")) or []
assert len(examples) >= 10, len(examples)
assert len(bqs) == 10, len(bqs)
print("PASS curated examples", len(examples), "benchmarks", len(bqs))
spark.sql(
    f"""CREATE OR REPLACE TABLE {cfg.table('agent_ids')} AS
    SELECT '{blank_id}' AS blank_space_id, '{curated_id}' AS curated_space_id, current_timestamp() AS updated_at"""
)
print("wrote", cfg.table("agent_ids"))
